# smol-ladder SFT (LoRA)

Runs `train/sft_lora.py` on Kaggle's 2x T4. Secrets come from Kaggle Secrets
(Add-ons -> Secrets); nothing is inlined.

Before the first run, set the kernel's `id` in `kernel-metadata.json` to your own
`user/slug` -- the placeholder here will not push. Then:

```sh
kaggle kernels push -p train/kaggle
```

Enable **Internet** (phone-verified) or the model cannot be downloaded. Select **T4**,
never P100: the Qwen3.5 architecture is unsupported on P100.

**Secrets**: `HF_TOKEN` (Hub access, and pushing the adapter), optionally
`WANDB_API_KEY`.

In [ ]:
# TRL pins transformers tightly; installing them together in one resolver pass is the
# only way to get a consistent set.
!pip install -q -U 'trl>=1.13' 'peft>=0.21' 'transformers>=5.17' 'accelerate>=1.0' 'datasets>=5.0' bitsandbytes

In [ ]:
import os
from kaggle_secrets import SecretsManager

secrets = SecretsManager()
# Internet on + the HF_TOKEN secret. Asserted rather than defaulted: without it the cell
# downloads a public model fine and then fails at the push, which costs a whole session.
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ.setdefault("HUB_MODEL_ID", secrets.get_secret("HUB_MODEL_ID"))

import torch
print("gpus", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Clone the repo

The Kaggle disk is wiped between sessions, so the repo is cloned fresh. `DATA` is not
available here: the ladder's table cache is a symlink to a machine that does not exist on
Kaggle, so this kernel trains on the **upstream SFT export only** (arm A). The other two
sources need the shared data dir.

In [ ]:
!git clone -q --depth 1 https://github.com/evan-kim2028/smol-ladder.git /kaggle/working/smol-ladder
%cd /kaggle/working/smol-ladder

## Export

`--heldout-columns task_id` because the SFT parquet carries `task_id` but no table
identifier, so the coarse `bucket_prefix` check cannot run on this source. The export
prints how many rows it refused; **read that number and record it** -- it is the difference
between a clean held-out evaluation and a contaminated one.

In [ ]:
!uv run --with datasets python -m train.export_sft \
    --source smoldataenvs-sft \
    --heldout-columns task_id \
    --out /kaggle/working/sft_data

## Train

Upstream's own hyperparameters, with `MAX_LENGTH=8192` (their trajectories are 3-12 turns).
One T4 is plenty for a 2B LoRA; the second is used by re-running the export/smoke cell, or
not at all for this stage.

`--bf16-compute` is off because these are T4s: bf16 matmul is emulated on sm_75 in many
torch builds and is much slower than fp16. Everything else matches upstream.

In [ ]:
!MODEL=Qwen/Qwen3.5-2B MAX_LENGTH=8192 BATCH_SIZE=1 GRAD_ACCUM=8 \
 LEARNING_RATE=2e-5 EPOCHS=1 \
 uv run --extra train python -m train.sft_lora \
    --data /kaggle/working/sft_data \
    --out /kaggle/working/sft_run \
    --max-length 8192 \
    --precision fp16 \
    --push-adapter $HUB_MODEL_ID

## Resuming

The 12 h session cap cuts runs off mid-epoch and the disk is wiped after it. Push to the
Hub **often** (`HUB_MODEL_ID` + `--push-adapter` sets `push_to_hub` with
`hub_strategy="checkpoint"`), and to resume add `--resume` plus a fresh clone: the
checkpoint has to be downloaded back from the Hub, because `/kaggle/working` is gone.

A 4,400-row export at batch 1 x accum 8 x 8192 tokens is roughly 550 optimizer steps and
fits in one session on one T4. The measured laptop rate is in `docs/TRAINING.md`; a T4 is
about 3-4x a power-limited 4050 on this workload.